In [1]:
import cv2


def prepare_image(image_path, target_size=(512, 512), to_grayscale=False):
    """
    Loads an image, resizes it to the target dimensions,
    and optionally converts it to grayscale.
    """
    # 1. Load the image from your file system
    image = cv2.imread(image_path)

    # Safety check: Did the image load correctly?
    if image is None:
        raise ValueError(f"Error: Could not load image at {image_path}. Check the file path.")

    # 2. Resize the image
    # cv2.INTER_AREA is the mathematical best method when shrinking an image
    resized_image = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)

    # 3. Convert to grayscale if requested
    if to_grayscale:
        # OpenCV loads images in BGR format by default, not RGB!
        final_image = cv2.cvtColor(resized_image, cv2.COLOR_BGR2GRAY)
    else:
        final_image = resized_image

    return final_image


# --- How you will use this in your main script ---

# Prepare the Brad Pitt image (Target) - We need this in grayscale to map brightness
brad_pitt_img = prepare_image('brad_pitt.jpg', target_size=(512, 512), to_grayscale=True)

# Prepare your random Source Image - We keep this in color!
source_img = prepare_image('my_random_picture.jpg', target_size=(512, 512), to_grayscale=False)

In [ ]:
import cv2
import numpy as np

def rearrange_pixels(source_color_img, target_gray_img):
    """
    Sorts pixels from the source image by brightness and maps them
    to the corresponding brightness locations in the target image.
    """
    # 1. Get the brightness map of your source image
    source_gray = cv2.cvtColor(source_color_img, cv2.COLOR_BGR2GRAY)

    # 2. Flatten all arrays from 2D grids into 1D lists
    # .reshape(-1, 3) keeps the BGR color channels intact while flattening the grid
    source_color_flat = source_color_img.reshape(-1, 3)
    source_gray_flat = source_gray.flatten()
    target_gray_flat = target_gray_img.flatten()

    # 3. Get the sorting indices using argsort()
    # This is the secret sauce: it doesn't just sort, it remembers the original X/Y address!
    source_sort_indices = np.argsort(source_gray_flat)
    target_sort_indices = np.argsort(target_gray_flat)

    # 4. Organize the source color pixels from darkest to lightest
    sorted_source_colors = source_color_flat[source_sort_indices]

    # 5. Create a blank 1D canvas to hold the final pixels
    output_flat = np.zeros_like(source_color_flat)

    # 6. Map the sorted source colors into the exact addresses of the target's shadows/highlights
    output_flat[target_sort_indices] = sorted_source_colors

    # 7. Reshape the 1D list back into a 512x512 2D image
    final_image = output_flat.reshape(source_color_img.shape)

    return final_image